# Piecewise Linear Regression — từ S0 đến S6

Mục tiêu: dự đoán `Post_Semester_GPA` từ 12 biến đầu vào. Notebook đi từ baseline tuyến tính S0 đến S6; mỗi bước nêu giả thuyết, cách tạo đặc trưng và kết quả để thấy vì sao bước tiếp theo được thử.

**Cách đọc kết quả:** RMSE càng thấp càng tốt. 5-fold CV chia riêng tập train thành 5 phần để đánh giá; validation là tập riêng dùng đối chiếu. Các model dùng cùng fold để so sánh. S3 chọn knot trên chính các fold này và nhiều ý tưởng được đề xuất sau khi xem kết quả development, nên CV/validation ở đây mang tính thăm dò, không phải đánh giá độc lập sau lựa chọn.

S6 được người thực hiện chọn làm model cuối và đã đánh giá trên test. Dữ liệu là synthetic; quan hệ dự đoán không chứng minh nhân quả. Thời điểm có sẵn các biến hành vi, Dependency và Anxiety chưa được xác minh.

Chọn kernel có dependencies trong `requirements.txt`, rồi Restart Kernel → Run All từ trong repo. Phần test mặc định không tự chạy lại. Kết quả run mới tạo thư mục riêng; lựa chọn S6 đã khóa không tự bị thay thế.


## 1. Cấu hình

Seed 42 và 5-fold CV giống phần interaction. Không tạo lại split. Output mỗi lần chạy nằm trong `outputs/piecewise/runs/<run_id>/`.

**Vì sao dùng cùng seed và folds?** Để sự khác nhau giữa các bước phản ánh chủ yếu thay đổi đặc trưng, thay vì do chia dữ liệu khác nhau. Random CV ở đây theo protocol của nhóm; nếu có nhiều quan sát cùng sinh viên hoặc dữ liệu theo thời gian, cần split theo nhóm hoặc thời gian phù hợp.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import uuid
import warnings

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.utils.validation import check_array, check_is_fitted
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'pyproject.toml').is_file() and (p / 'data/train/dataset.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Mở notebook bên trong repo có data/train/dataset.csv.')
DATA_ROOT = ROOT / 'data'
SEED = 42
N_FOLDS = 5
TARGET = 'Post_Semester_GPA'
NUMERIC = ['Weekly_GenAI_Hours', 'Pre_Semester_GPA', 'Traditional_Study_Hours',
           'Tool_Diversity', 'Paid_Subscription',
           'Perceived_AI_Dependency', 'Anxiety_Level_During_Exams', 'Year_of_Study', 'Institutional_Policy']
YEAR_LEVELS = {'Freshman': 1, 'Sophomore': 2, 'Junior': 3, 'Senior': 4, 'Graduate': 5}
POLICY_LEVELS = {'Strict_Ban': 0, 'Allowed_With_Citation': 1, 'Actively_Encouraged': 2}
CATEGORICAL = ['Major_Category', 'Primary_Use_Case',
               'Prompt_Engineering_Skill']
INPUTS = NUMERIC + CATEGORICAL
print('Repo:', ROOT)

## 2. Đọc frozen splits và kiểm tra dữ liệu

Đối chiếu SHA-256 với metadata, kiểm tra membership theo assignment và ID không trùng trong/giữa các tập. Không đọc features hoặc target test. Thiếu predictor được xử lý trong pipeline; target thiếu/không hợp lệ làm dừng thực thi. ID và các outcome phụ không làm predictor.

**Vì sao kiểm tra split?** Đảm bảo không có sinh viên trùng giữa các tập và dữ liệu không đổi so với split của nhóm. Chỉ đọc ID test để kiểm tra membership, không dùng giá trị test khi tạo đặc trưng.


In [ ]:
metadata = json.loads((DATA_ROOT / 'metadata.json').read_text(encoding='utf-8'))
assignment_path = DATA_ROOT / 'assignments.csv'
if hashlib.sha256(assignment_path.read_bytes()).hexdigest() != metadata['assignment_sha256']:
    raise ValueError('Assignment hash không khớp metadata.')
assignment = pd.read_csv(assignment_path, dtype={'Student_ID': str})
if assignment['Student_ID'].isna().any() or not assignment['Student_ID'].is_unique:
    raise ValueError('Assignment có ID thiếu hoặc trùng.')
frames, hashes, seen = {}, {}, set()
for split in ['train', 'validation', 'test']:
    path = DATA_ROOT / split / 'dataset.csv'
    hashes[split] = hashlib.sha256(path.read_bytes()).hexdigest()
    if hashes[split] != metadata['dataset_sha256'][f'{split}/dataset.csv']:
        raise ValueError(f'{split}: CSV hash không khớp metadata; không ghi đè split.')
    frame = pd.read_csv(path, dtype={'Student_ID': str},
                        usecols=['Student_ID'] if split == 'test' else None)
    ids = frame['Student_ID']
    if ids.isna().any() or not ids.is_unique or seen.intersection(ids):
        raise ValueError(f'{split}: ID thiếu hoặc trùng trong/giữa các tập.')
    if set(ids) != set(assignment.loc[assignment['split'] == split, 'Student_ID']):
        raise ValueError(f'{split}: ID membership khác assignment.')
    seen.update(ids)
    frames[split] = frame
if seen != set(assignment['Student_ID']):
    raise ValueError('Split không phủ hết assignment.')

train, val = frames['train'], frames['validation']
for split, frame in [('train', train), ('validation', val)]:
    missing = sorted(set(INPUTS + [TARGET]) - set(frame.columns))
    if missing:
        raise ValueError(f'{split}: thiếu cột {missing}')
    unknown_years = set(frame['Year_of_Study'].dropna()) - set(YEAR_LEVELS)
    if unknown_years:
        raise ValueError(f'{split}: Year_of_Study không xác định: {unknown_years}')
    frame['Year_of_Study'] = frame['Year_of_Study'].map(YEAR_LEVELS)
    unknown_policies = set(frame['Institutional_Policy'].dropna()) - set(POLICY_LEVELS)
    if unknown_policies:
        raise ValueError(f'{split}: Institutional_Policy không xác định: {unknown_policies}')
    frame['Institutional_Policy'] = frame['Institutional_Policy'].map(POLICY_LEVELS)
    for column in NUMERIC + [TARGET]:
        frame[column] = pd.to_numeric(frame[column], errors='raise')
        if np.isinf(frame[column].to_numpy(dtype=float)).any():
            raise ValueError(f'{split}: {column} có infinity.')
    if frame[TARGET].isna().any() or not frame[TARGET].between(0, 4).all():
        raise ValueError(f'{split}: target thiếu hoặc ngoài [0, 4].')
    if not frame['Paid_Subscription'].dropna().isin([0, 1]).all():
        raise ValueError(f'{split}: Paid_Subscription phải là boolean hoặc 0/1.')
    for column in CATEGORICAL:
        frame[column] = frame[column].map(lambda x: str(x) if pd.notna(x) else np.nan)
    if frame[INPUTS].isna().all().any():
        raise ValueError(f'{split}: có predictor thiếu toàn bộ.')

display(pd.DataFrame({'rows': {name: len(frame) for name, frame in frames.items()}}))
display(train[INPUTS].isna().sum().rename('train_missing').to_frame())

### 2.1. Từng biến numeric với Post-Semester GPA — train

**X nguyên rời rạc:** nhóm sinh viên theo từng giá trị X, tính mean Post_Semester_GPA cho mỗi nhóm và nối các điểm thành đường. Nhãn n cho biết số mẫu mỗi điểm. Mean là GPA trung bình của nhóm, không phải mean của X. Đường nối chỉ giúp quan sát, không khẳng định có giá trị trung gian giữa các mức rời rạc.

**X liên tục:** giữ scatter plot, mỗi chấm là một sinh viên; Y là Post_Semester_GPA thực tế.

Dùng toàn bộ cặp không thiếu của train; không impute cho EDA. Paid_Subscription: False=0, True=1. Year: Freshman=1, Sophomore=2, Junior=3, Senior=4, Graduate=5. Policy: Strict_Ban=0, Allowed_With_Citation=1, Actively_Encouraged=2.

Đây là quan hệ hai biến quan sát, chưa kiểm soát inputs khác; không phải đường dự đoán S0–S4 và không chứng minh nhân quả. Các hình mean dùng trục Y tự động để thấy chênh lệch; xem giá trị trục khi so sánh.


In [ ]:
numeric_figures = {}
numeric_plot_rows = []
numeric_group_means = {}
for column in NUMERIC:
    pairs = train[[column, TARGET]].dropna()
    if pairs.empty:
        raise ValueError(f'{column}: không có cặp dữ liệu để vẽ.')
    integer_x = bool(np.equal(pairs[column].to_numpy(), np.floor(pairs[column].to_numpy())).all())
    fig, ax = plt.subplots(figsize=(7, 4.5))
    if integer_x:
        grouped = pairs.groupby(column, sort=True)[TARGET].agg(mean='mean', n='count').reset_index()
        assert grouped['n'].sum() == len(pairs)
        assert np.isclose(np.average(grouped['mean'], weights=grouped['n']), pairs[TARGET].mean())
        numeric_group_means[column] = grouped
        line, = ax.plot(grouped[column], grouped['mean'], marker='o', color='#2563eb', linewidth=1.8)
        np.testing.assert_allclose(line.get_ydata(), grouped['mean'])
        for row in grouped.itertuples(index=False, name=None):
            level, mean, count = row
            ax.annotate(f'n={int(count):,}', (level, mean), xytext=(0, 9),
                        textcoords='offset points', ha='center', fontsize=8)
        ax.margins(x=0.12, y=0.25)
        ax.set_xticks(grouped[column])
        ylabel = 'Mean Post_Semester_GPA'
        plot_type = 'group_mean_line'
    else:
        points = ax.scatter(pairs[column], pairs[TARGET], s=5, alpha=0.08,
                            color='#2563eb', edgecolors='none', rasterized=True)
        assert len(points.get_offsets()) == len(pairs)
        ax.set_ylim(-0.05, 4.05)
        ylabel = TARGET
        plot_type = 'scatter'
    ax.set(xlabel=column, ylabel=ylabel,
           title=f'{column} vs {ylabel}\nTrain · n={len(pairs):,}')
    ax.grid(alpha=0.2)
    ax.set_axisbelow(True)
    fig.tight_layout()
    numeric_figures[column] = fig
    numeric_plot_rows.append({'feature': column, 'plot_type': plot_type, 'n_pairs': len(pairs),
        'missing_pairs': len(train) - len(pairs), 'scope': 'train',
        'file': f'numeric_vs_gpa/{column}.png'})
    display(fig)
    plt.close(fig)
assert set(numeric_figures) == set(NUMERIC)
numeric_plot_manifest = pd.DataFrame(numeric_plot_rows)
display(numeric_plot_manifest)


## 3. S0: preprocessing → OLS

Numeric: median imputation. Categorical: most-frequent imputation và One-Hot Encoding, bỏ mức reference bằng `drop='first'`. Category mới được cảnh báo và mã hóa toàn 0, giống reference.

Mọi thông số được fit lại trong từng training fold. Không scale để giữ đơn vị gốc; chưa centering, hinge hay interaction. Với đủ category levels như dữ liệu nhóm, S0 có 19 predictor terms (không tính intercept).

`Perceived_AI_Dependency` và `Anxiety_Level_During_Exams` được dùng như numeric scores: mỗi biến có một hệ số, giả định mức thay đổi dự đoán không đổi khi tăng một điểm, giữ các biến khác cố định. Đây là giả định đơn giản hóa cho thang điểm ordinal, cần đánh giá bằng CV; không mặc định tốt hơn one-hot.

S0–S2 dùng chung encoding này. Số terms khác baseline interaction dùng one-hot cho Dependency, Anxiety, Year và Policy, nên chênh lệch kết quả giữa hai phần còn có thể do encoding.

`Year_of_Study`: Freshman=1, Sophomore=2, Junior=3, Senior=4, Graduate=5. Đây là mã thứ tự, không phải số năm học thực tế của Graduate. OLS giả định cùng mức thay đổi dự đoán giữa hai mức liên tiếp.

`Institutional_Policy`: Strict_Ban=0, Allowed_With_Citation=1, Actively_Encouraged=2. Dùng như numeric áp đặt cùng mức thay đổi dự đoán từ 0→1 và 1→2; đây là giả định mô hình, không phải khoảng cách chính sách đã được đo lường.

**Lý do chọn baseline:** OLS dễ diễn giải và cho mốc so sánh trước khi thêm độ phức tạp. Median và mức phổ biến xử lý giá trị thiếu; one-hot phù hợp với loại không có thứ tự, còn bỏ một mức giúp tránh dư thừa tuyến tính với intercept. Không scale vì OLS không regularization không cần scale để tạo dự đoán, và hệ số giữ đơn vị gốc. Các mã ordinal ở trên là lựa chọn đơn giản theo yêu cầu, chưa được chứng minh tốt hơn one-hot.


In [ ]:
preprocessing = ColumnTransformer([
    ('numeric', SimpleImputer(strategy='median'), NUMERIC),
    ('categorical', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encode', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)),
    ]), CATEGORICAL),
], remainder='drop')
s0 = Pipeline([('preprocessing', preprocessing), ('ols', LinearRegression())])

def metrics(y_true, prediction):
    mse = mean_squared_error(y_true, prediction)
    return {'mae': float(mean_absolute_error(y_true, prediction)),
            'mse': float(mse), 'rmse': float(np.sqrt(mse)),
            'r2': float(r2_score(y_true, prediction))}

## 4. Kiểm tra nhỏ có thể chạy lại

Dùng vài hàng train làm fixture để kiểm chứng median chỉ học từ dữ liệu fit, category reference và khả năng dự đoán. Việc gọi `predict()` không được thay đổi median đã học.


In [ ]:
fixture = train[INPUTS].head(6).copy()
fixture['Weekly_GenAI_Hours'] = [0., 2., np.nan, 4., 6., 8.]
for column in CATEGORICAL:
    fixture[column] = ['A', 'B', 'A', 'B', 'A', 'B']
assert pd.Series(['Strict_Ban', 'Allowed_With_Citation', 'Actively_Encouraged']).map(POLICY_LEVELS).tolist() == [0, 1, 2]
check = clone(s0).fit(fixture, np.arange(6, dtype=float))
prep = check.named_steps['preprocessing']
medians = prep.named_transformers_['numeric'].statistics_.copy()
assert medians[0] == 4.0
names = prep.get_feature_names_out()
assert len(names) == len(NUMERIC) + len(CATEGORICAL)
for column in ['Perceived_AI_Dependency', 'Anxiety_Level_During_Exams', 'Year_of_Study', 'Institutional_Policy']:
    assert list(names).count('numeric__' + column) == 1
    index = list(names).index('numeric__' + column)
    np.testing.assert_allclose(prep.transform(fixture)[:, index], fixture[column].to_numpy(dtype=float))
new = fixture.iloc[[0]].copy()
new['Weekly_GenAI_Hours'] = np.nan
transformed = prep.transform(new)
assert transformed[0, 0] == 4.0
assert np.all(transformed[0, len(NUMERIC):] == 0)
assert np.isfinite(check.predict(new)).all()
np.testing.assert_array_equal(prep.named_transformers_['numeric'].statistics_, medians)
print('PASS: train-only median, reference coding, finite prediction.')

## 5. Đánh giá 5-fold CV trên train

Cùng `KFold(shuffle=True, random_state=42)` với interaction. Lưu fold assignment để S1–S2 dùng cùng rows. Pipeline được clone và fit riêng mỗi fold; validation không tham gia fit.

**Vì sao CV?** Mỗi lần học trên 4/5 train và tính lỗi trên 1/5 còn lại, lặp đủ 5 lần. Trung bình RMSE tóm tắt lỗi; độ lệch chuẩn cho biết mức thay đổi giữa các fold. Bảng fold ghép cặp phía sau giúp kiểm tra cải thiện có nhất quán hay không.


In [ ]:
X, y = train[INPUTS], train[TARGET]
folds = list(KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(X))
fold_ids = np.empty(len(train), dtype=int)
records = []
for fold, (fit_idx, score_idx) in enumerate(folds, start=1):
    fold_model = clone(s0).fit(X.iloc[fit_idx], y.iloc[fit_idx])
    prediction = fold_model.predict(X.iloc[score_idx])
    records.append({'model': 'S0', 'fold': fold, 'n': len(score_idx),
                    **metrics(y.iloc[score_idx], prediction)})
    fold_ids[score_idx] = fold
cv_results = pd.DataFrame(records)
cv_summary = cv_results[['mae', 'mse', 'rmse', 'r2']].agg(['mean', 'std']).T
cv_summary.columns = ['cv_mean', 'cv_sd']
display(cv_results)
display(cv_summary)

## 6. Fit trên train và đối chiếu validation

Model S0 được fit trên toàn bộ train. Validation chỉ dùng đối chiếu, không refit. Bảng hệ số thể hiện quan hệ có điều kiện; chưa có kết luận về ý nghĩa thống kê hay nhân quả. Dự đoán giữ nguyên, không clip về [0, 4].


In [ ]:
s0.fit(X, y)
val_prediction = s0.predict(val[INPUTS])
validation_metrics = metrics(val[TARGET], val_prediction)
feature_names = s0.named_steps['preprocessing'].get_feature_names_out()
coefficients = pd.DataFrame({'feature': ['intercept', *feature_names],
                             'coefficient': [s0.named_steps['ols'].intercept_,
                                             *s0.named_steps['ols'].coef_]})
assert np.isfinite(val_prediction).all()
assert len(feature_names) == len(NUMERIC) + sum(train[c].nunique() - 1 for c in CATEGORICAL)
print('S0 predictor terms:', len(feature_names), '(không gồm intercept)')
display(pd.Series(validation_metrics, name='validation').to_frame())
display(coefficients)
print('Dự đoán ngoài [0, 4]:', int(((val_prediction < 0) | (val_prediction > 4)).sum()))
print('Test không được chạy trong bước development này.')

## 7. S1: thêm một hinge AI Hours

H = Weekly_GenAI_Hours; knot k = median H của phần training đang fit.

**hinge = max(0, H − k)**

**S1 = main effects S0 + γ × hinge**

Trước knot: độ dốc là βH; sau knot: βH + γ. Giữ H gốc nên hai đoạn nối liên tục tại k. S1 có 20 predictor terms khi đủ levels.

Pipeline: preprocessing như S0 → thêm hinge → OLS. AI Hours là cột đầu của ma trận preprocessing. Median được học sau imputation, vẫn bằng median H không thiếu của training data. Validation và phần đánh giá CV chỉ dùng lại knot, không học knot mới.

**Vì sao thử bước này?** Một đường thẳng của S0 giả định độ dốc theo số giờ AI không đổi. Median là mốc đầu tiên đơn giản, chia dữ liệu thành hai phần tương đối cân bằng. Nó không khẳng định đây là ngưỡng tự nhiên. Trong run đã lưu, CV RMSE giảm từ 0,160907 xuống 0,159714 và validation từ 0,161136 xuống 0,159768; vì vậy đáng thử thêm độ linh hoạt.


In [ ]:
class QuantileHinges(BaseEstimator, TransformerMixin):
    """Append feature hinges using quantiles learned on the fit matrix."""
    def __init__(self, quantiles=(0.5,), feature_index=0):
        self.quantiles = quantiles
        self.feature_index = feature_index

    def fit(self, X, y=None):
        X = check_array(X, dtype=float)
        q = np.asarray(self.quantiles, dtype=float)
        if q.ndim != 1 or len(q) == 0 or not np.all((q > 0) & (q < 1)) or np.any(np.diff(q) <= 0):
            raise ValueError('Quantiles phải tăng nghiêm ngặt và nằm trong (0, 1).')
        self.n_features_in_ = X.shape[1]
        # ponytail: chỉ số theo thứ tự preprocessing; dùng tên cột nếu đổi bố cục.
        if not isinstance(self.feature_index, int) or not 0 <= self.feature_index < X.shape[1]:
            raise ValueError('feature_index nằm ngoài ma trận đầu vào.')
        self.knots_ = np.quantile(X[:, self.feature_index], q)
        if len(np.unique(self.knots_)) != len(q):
            raise ValueError('Quantile knots trùng nhau; giảm số knots trước khi đánh giá.')
        return self

    def transform(self, X):
        check_is_fitted(self, 'knots_')
        X = check_array(X, dtype=float)
        if X.shape[1] != self.n_features_in_:
            raise ValueError('Số cột đầu vào hinge khác dữ liệu fit.')
        return np.column_stack([X, np.maximum(0, X[:, [self.feature_index]] - self.knots_)])

s1 = Pipeline([('preprocessing', clone(preprocessing)),
               ('hinge', QuantileHinges()), ('ols', LinearRegression())])


### 7.1. Kiểm tra hinge và tính độc lập với dữ liệu đánh giá

Fixture có knot = 6: H=4 → 0; H=6 → 0; H=8 → 2. Kiểm tra cả imputation trong pipeline, giữ nguyên các main effects và không đổi knot khi transform batch mới.


In [ ]:
toy_design = np.array([[0., 10.], [4., 20.], [6., 30.], [8., 40.], [12., 50.]])
hinge_check = QuantileHinges().fit(toy_design)
assert hinge_check.knots_[0] == 6.0
z = hinge_check.transform(toy_design)
np.testing.assert_array_equal(z[:, :-1], toy_design)
np.testing.assert_array_equal(z[:, -1], [0., 0., 0., 2., 6.])
np.testing.assert_array_equal(
    hinge_check.transform(np.array([[4., 0.], [6., 0.], [8., 0.], [100., 0.]]))[:, -1],
    [0., 0., 2., 94.])
assert hinge_check.knots_[0] == 6.0
pipeline_check = clone(s1).fit(fixture, np.arange(len(fixture), dtype=float))
assert pipeline_check.named_steps['hinge'].knots_[0] == 4.0
imputed = pipeline_check.named_steps['preprocessing'].transform(new)
assert pipeline_check.named_steps['hinge'].transform(imputed)[0, -1] == 0.0
assert pipeline_check.named_steps['hinge'].n_features_in_ == len(NUMERIC) + len(CATEGORICAL)
assert np.isfinite(pipeline_check.predict(new)).all()
print('PASS: hinge values, unchanged main effects, fixed train-only knot, missing H.')

## 8. S1: cùng CV folds, fit train và đối chiếu validation

Fit lại toàn bộ pipeline S1 trong từng fold. Ghi knot từng fold để kiểm chứng phạm vi học. Sau CV, fit S1 trên toàn bộ train; không fit validation. Bảng S0–S1 dùng cùng folds và cùng validation rows; chênh lệch RMSE nhỏ chưa đủ khẳng định cải thiện ổn định.


In [ ]:
s1_records = []
for fold, (fit_idx, score_idx) in enumerate(folds, start=1):
    fold_model = clone(s1).fit(X.iloc[fit_idx], y.iloc[fit_idx])
    knot = fold_model.named_steps['hinge'].knots_[0]
    assert np.isclose(knot, X.iloc[fit_idx]['Weekly_GenAI_Hours'].median())
    prediction = fold_model.predict(X.iloc[score_idx])
    s1_records.append({'model': 'S1', 'fold': fold, 'n': len(score_idx), 'knot': knot,
                       **metrics(y.iloc[score_idx], prediction)})
s1_cv_results = pd.DataFrame(s1_records)
s1_cv_summary = s1_cv_results[['mae', 'mse', 'rmse', 'r2']].agg(['mean', 'std']).T
s1_cv_summary.columns = ['cv_mean', 'cv_sd']

s1.fit(X, y)
s1_val_prediction = s1.predict(val[INPUTS])
s1_validation_metrics = metrics(val[TARGET], s1_val_prediction)
s1_feature_names = [*s1.named_steps['preprocessing'].get_feature_names_out(), 'hinge__AI_Hours']
assert len(s1_feature_names) == len(feature_names) + 1
assert np.isfinite(s1_val_prediction).all()
s1_coefficients = pd.DataFrame({'feature': ['intercept', *s1_feature_names],
                               'coefficient': [s1.named_steps['ols'].intercept_,
                                               *s1.named_steps['ols'].coef_]})
s1_knot = s1.named_steps['hinge'].knots_[0]
beta_h = float(s1.named_steps['ols'].coef_[0])
gamma = float(s1.named_steps['ols'].coef_[-1])
s1_slopes = {'knot': s1_knot, 'before_knot': beta_h,
             'after_knot': beta_h + gamma, 'slope_change': gamma}

comparison_rows = []
for name, summary, scores, terms in [
    ('S0', cv_summary, validation_metrics, len(feature_names)),
    ('S1', s1_cv_summary, s1_validation_metrics, len(s1_feature_names)),
]:
    comparison_rows.append({'model': name, 'terms': terms,
        **{f'cv_{metric}_mean': float(summary.loc[metric, 'cv_mean']) for metric in summary.index},
        **{f'cv_{metric}_sd': float(summary.loc[metric, 'cv_sd']) for metric in summary.index},
        **{f'val_{metric}': value for metric, value in scores.items()}})
comparison = pd.DataFrame(comparison_rows)
display(s1_cv_results)
display(comparison)
display(pd.Series(s1_slopes, name='S1 train-fit').to_frame())
display(s1_coefficients)
delta = float(s1_cv_summary.loc['rmse', 'cv_mean'] - cv_summary.loc['rmse', 'cv_mean'])
print(f'CV RMSE S1 − S0 = {delta:+.6f} (âm là S1 thấp hơn).')
print('S1 validation predictions ngoài [0, 4]:', int(((s1_val_prediction < 0) | (s1_val_prediction > 4)).sum()))
print('Độ dốc là quan hệ dự đoán khi giữ các input khác cố định, không phải tác động nhân quả.')
print('Test không được chạy trong bước development này.')

## 9. S2: ba hinges và bốn đoạn thẳng

H = AI Hours; k₁, k₂, k₃ lần lượt là quantile 25%, 50%, 75% của H sau imputation trên phần training đang fit.

**z₁ = max(0, H − k₁); z₂ = max(0, H − k₂); z₃ = max(0, H − k₃)**

S2 giữ 19 main-effect terms của S0 và thêm 3 hinges: tổng 22 predictor terms. Độ dốc lần lượt là βH; βH+γ₁; βH+γ₁+γ₂; βH+γ₁+γ₂+γ₃. Các đoạn nối liên tục.

Dùng lại `QuantileHinges`, với ba quantiles thay vì một median. Khi H thiếu, preprocessing điền median trước khi học quantiles và tạo hinges. Nếu có nhiều missing, quantiles sau imputation có thể khác quantiles chỉ trên giá trị không thiếu; notebook không dùng validation để tính các mốc này.

**Vì sao thử ba knot?** Sau S1, các quantile 25%, 50%, 75% cho phép bốn đoạn và tránh tự chọn ngưỡng theo test. S2 chỉ giảm CV RMSE từ 0,159714 xuống 0,159522; validation từ 0,159768 xuống 0,159631. Mức giảm nhỏ, nên việc giữ S2 cho các bước sau là lựa chọn thăm dò, chưa chứng minh cần ba knot hơn một knot.


In [ ]:
s2 = Pipeline([('preprocessing', clone(preprocessing)),
               ('hinge', QuantileHinges(quantiles=(0.25, 0.5, 0.75))),
               ('ols', LinearRegression())])

# Một runnable check: knots [4, 6, 8], giữ nguyên main effects.
three = QuantileHinges(quantiles=(0.25, 0.5, 0.75)).fit(toy_design)
np.testing.assert_array_equal(three.knots_, [4., 6., 8.])
z = three.transform(toy_design)
np.testing.assert_array_equal(z[:, :-3], toy_design)
np.testing.assert_array_equal(z[:, -3:], [[0., 0., 0.], [0., 0., 0.],
                                        [2., 0., 0.], [4., 2., 0.], [8., 6., 4.]])
# Batch mới có phân phối khác không được thay đổi knots.
three.transform(np.array([[100., 0.], [200., 0.]]))
np.testing.assert_array_equal(three.knots_, [4., 6., 8.])
# Hai biểu thức ở hai phía mỗi knot phải gặp nhau tại knot.
beta, gammas = 0.02, np.array([0.01, -0.02, 0.03])
for j, knot in enumerate(three.knots_):
    left = beta * knot + np.dot(gammas[:j], knot - three.knots_[:j])
    right = beta * knot + np.dot(gammas[:j+1], knot - three.knots_[:j+1])
    assert np.isclose(left, right)
try:
    QuantileHinges(quantiles=(0.25, 0.5, 0.75)).fit(np.ones((5, 2)))
except ValueError as exc:
    assert 'trùng nhau' in str(exc)
else:
    raise AssertionError('Phải báo lỗi khi ba knots trùng nhau.')
print('PASS: three hinge values, unchanged main effects, fixed knots, continuity, duplicate knots.')

## 10. S2: CV và so sánh S0–S2

Cùng train folds, preprocessing và validation rows như S0/S1. Lưu ba knots mỗi fold. Fit toàn bộ train sau CV để lấy model và bốn độ dốc; không dùng test trong bước này. Không mặc định nhiều hinges sẽ tốt hơn.


In [ ]:
s2_records = []
for fold, (fit_idx, score_idx) in enumerate(folds, start=1):
    fold_model = clone(s2).fit(X.iloc[fit_idx], y.iloc[fit_idx])
    knots = fold_model.named_steps['hinge'].knots_
    fit_design = fold_model.named_steps['preprocessing'].transform(X.iloc[fit_idx])
    np.testing.assert_allclose(knots, np.quantile(fit_design[:, 0], [0.25, 0.5, 0.75]))
    prediction = fold_model.predict(X.iloc[score_idx])
    s2_records.append({'model': 'S2', 'fold': fold, 'n': len(score_idx),
        **{f'knot_{j}': float(k) for j, k in enumerate(knots, start=1)},
        **metrics(y.iloc[score_idx], prediction)})
s2_cv_results = pd.DataFrame(s2_records)
s2_cv_summary = s2_cv_results[['mae', 'mse', 'rmse', 'r2']].agg(['mean', 'std']).T
s2_cv_summary.columns = ['cv_mean', 'cv_sd']
s2.fit(X, y)
s2_val_prediction = s2.predict(val[INPUTS])
s2_validation_metrics = metrics(val[TARGET], s2_val_prediction)
s2_feature_names = [*s2.named_steps['preprocessing'].get_feature_names_out(),
                    'hinge__AI_Hours_q25', 'hinge__AI_Hours_q50', 'hinge__AI_Hours_q75']
assert len(s2_feature_names) == len(feature_names) + 3
assert np.isfinite(s2_val_prediction).all()
s2_coefficients = pd.DataFrame({'feature': ['intercept', *s2_feature_names],
    'coefficient': [s2.named_steps['ols'].intercept_, *s2.named_steps['ols'].coef_]})
s2_knots = s2.named_steps['hinge'].knots_.tolist()
s2_gamma = s2.named_steps['ols'].coef_[-3:]
s2_segment_slopes = (s2.named_steps['ols'].coef_[0] + np.r_[0., np.cumsum(s2_gamma)]).tolist()
s2_hinge_details = {'quantiles': [0.25, 0.5, 0.75], 'knots': s2_knots,
    'slope_changes': s2_gamma.tolist(), 'segment_slopes': s2_segment_slopes}
comparison = pd.concat([comparison, pd.DataFrame([{'model': 'S2', 'terms': len(s2_feature_names),
    **{f'cv_{metric}_mean': float(s2_cv_summary.loc[metric, 'cv_mean']) for metric in s2_cv_summary.index},
    **{f'cv_{metric}_sd': float(s2_cv_summary.loc[metric, 'cv_sd']) for metric in s2_cv_summary.index},
    **{f'val_{metric}': value for metric, value in s2_validation_metrics.items()}}])], ignore_index=True)
display(s2_cv_results)
display(comparison)
print('S2 train-fit knots:', s2_knots)
display(pd.DataFrame({'segment': ['H ≤ k1', 'k1 < H ≤ k2', 'k2 < H ≤ k3', 'H > k3'],
                      'slope_GPA_per_hour': s2_segment_slopes}))
display(s2_coefficients)
print('S2 validation predictions ngoài [0, 4]:', int(((s2_val_prediction < 0) | (s2_val_prediction > 4)).sum()))
print('Test không được chạy trong bước development này.')

## 11. S3: thêm một hinge Pre GPA, chọn knot bằng train CV

**S3 = S2 + γP × max(0, P − kP)**, P là Pre_Semester_GPA.

Giữ nguyên ba hinges AI Hours; chỉ thêm một hinge Pre GPA. Thử quantile 50%, 75%, 90% (khai báo trước khi chạy), chọn quantile có mean CV RMSE thấp nhất; nếu bằng nhau chọn quantile nhỏ hơn. Trong mỗi fold, giá trị knot được học riêng sau imputation từ training portion. Validation chỉ đánh giá cấu hình đã chọn, không dùng để chọn knot.

S3 có 23 predictor terms. Độ dốc theo Pre GPA trước knot là βP, sau knot là βP+γP. Hinge không tự chặn GPA dự đoán ở 4 và không chứng minh ceiling effect. Đây là thí nghiệm mở rộng sau khi đã xem development results; không dùng test trong bước này.

**Vì sao thêm Pre GPA?** Điểm trước học kỳ là ứng viên có quan hệ mạnh với điểm sau học kỳ; một hinge cho phép độ dốc khác ở vùng điểm cao. Đây là giả thuyết cần kiểm tra, không mặc định có hiệu ứng trần. Run đã lưu chọn quantile 75% vì CV RMSE 0,157425 thấp hơn 0,158265 (50%) và 0,157505 (90%); validation S3 là 0,157403 so với S2 0,159631. Chênh lệch 75% và 90% nhỏ, và điểm CV dùng để chọn knot có thể lạc quan.


In [ ]:
P_INDEX = NUMERIC.index('Pre_Semester_GPA')
S3_QUANTILES = (0.5, 0.75, 0.9)

# Reuse transformer: cột 1 là Pre GPA, các hinges AI Hours nằm ở cuối.
pre_gpa_toy = np.array([[1., 2.], [2., 2.5], [3., 3.], [4., 3.5], [5., 4.]])
p_check = QuantileHinges(feature_index=P_INDEX).fit(pre_gpa_toy)
assert p_check.knots_[0] == 3.0
np.testing.assert_allclose(p_check.transform(pre_gpa_toy)[:, -1], [0., 0., 0., 0.5, 1.])
np.testing.assert_array_equal(p_check.transform(pre_gpa_toy)[:, :-1], pre_gpa_toy)
p_check.transform(np.array([[100., 4.]]))
assert p_check.knots_[0] == 3.0
print('PASS: Pre GPA hinge selects correct column; fixed knot, unchanged main effects.')

s3_records = []
for quantile in S3_QUANTILES:
    candidate = Pipeline([('preprocessing', clone(preprocessing)),
        ('hinge', QuantileHinges(quantiles=(0.25, 0.5, 0.75))),
        ('pre_gpa_hinge', QuantileHinges(quantiles=(quantile,), feature_index=P_INDEX)),
        ('ols', LinearRegression())])
    for fold, (fit_idx, score_idx) in enumerate(folds, start=1):
        fitted = clone(candidate).fit(X.iloc[fit_idx], y.iloc[fit_idx])
        fit_design = fitted.named_steps['preprocessing'].transform(X.iloc[fit_idx])
        p_knot = float(fitted.named_steps['pre_gpa_hinge'].knots_[0])
        assert np.isclose(p_knot, np.quantile(fit_design[:, P_INDEX], quantile))
        np.testing.assert_allclose(fitted.named_steps['hinge'].knots_,
                                  np.quantile(fit_design[:, 0], [0.25, 0.5, 0.75]))
        prediction = fitted.predict(X.iloc[score_idx])
        s3_records.append({'model': 'S3', 'quantile': quantile, 'fold': fold,
            'n': len(score_idx), 'pre_gpa_knot': p_knot,
            **{f'hours_knot_{j}': float(k) for j,k in enumerate(fitted.named_steps['hinge'].knots_,1)},
            **metrics(y.iloc[score_idx], prediction)})
s3_candidate_folds = pd.DataFrame(s3_records)
s3_knot_search = s3_candidate_folds.groupby('quantile')['rmse'].agg(['mean', 'std']).reset_index()
s3_quantile = float(s3_knot_search.sort_values(['mean', 'quantile']).iloc[0]['quantile'])
s3_cv_results = s3_candidate_folds[s3_candidate_folds['quantile'] == s3_quantile].copy()
s3_cv_summary = s3_cv_results[['mae', 'mse', 'rmse', 'r2']].agg(['mean', 'std']).T
s3_cv_summary.columns = ['cv_mean', 'cv_sd']
s3 = Pipeline([('preprocessing', clone(preprocessing)),
    ('hinge', QuantileHinges(quantiles=(0.25, 0.5, 0.75))),
    ('pre_gpa_hinge', QuantileHinges(quantiles=(s3_quantile,), feature_index=P_INDEX)),
    ('ols', LinearRegression())]).fit(X, y)
s3_val_prediction = s3.predict(val[INPUTS])
s3_validation_metrics = metrics(val[TARGET], s3_val_prediction)
s3_feature_names = [*s2_feature_names, 'hinge__Pre_GPA']
assert len(s3_feature_names) == len(feature_names) + 4
assert np.isfinite(s3_val_prediction).all()
s3_coefficients = pd.DataFrame({'feature': ['intercept', *s3_feature_names],
    'coefficient': [s3.named_steps['ols'].intercept_, *s3.named_steps['ols'].coef_]})
s3_p_knot = float(s3.named_steps['pre_gpa_hinge'].knots_[0])
s3_beta_p = float(s3.named_steps['ols'].coef_[P_INDEX])
s3_gamma_p = float(s3.named_steps['ols'].coef_[-1])
s3_hinge_details = {'pre_gpa_quantile': s3_quantile, 'pre_gpa_knot': s3_p_knot,
    'pre_gpa_slope_before': s3_beta_p, 'pre_gpa_slope_after': s3_beta_p + s3_gamma_p,
    'pre_gpa_slope_change': s3_gamma_p, 'hours_knots': s3.named_steps['hinge'].knots_.tolist()}
comparison = pd.concat([comparison, pd.DataFrame([{'model': 'S3', 'terms': len(s3_feature_names),
    **{f'cv_{metric}_mean': float(s3_cv_summary.loc[metric, 'cv_mean']) for metric in s3_cv_summary.index},
    **{f'cv_{metric}_sd': float(s3_cv_summary.loc[metric, 'cv_sd']) for metric in s3_cv_summary.index},
    **{f'val_{metric}': value for metric, value in s3_validation_metrics.items()}}])], ignore_index=True)
display(s3_knot_search)
display(comparison)
display(pd.Series(s3_hinge_details, name='S3 train fit').to_frame())
print('CV của cấu hình thắng đã được dùng chọn knot, không phải ước lượng độc lập sau tuning.')
print('Test không được chạy trong bước development này.')

## 12. S4: AI Hours spline × Use Case

Giữ S3 và thêm **H×U, z₁×U, z₂×U, z₃×U** cho mỗi dummy Use Case. Với 5 Use Cases có 4 dummies: thêm 16 terms, tổng 39 predictor terms khi đủ levels. Không nhân hinge Pre GPA với Use Case.

Mỗi nhóm có thể có độ dốc khác nhau ở bốn khoảng nhưng dùng chung knots. Giữ đủ main effects và hinges để bảo toàn hierarchy; các đường liên tục tại knots.

Giữ quantile Pre GPA đã được train CV chọn cho S3, không tuning thêm cho S4. Trong mỗi fold, fit lại preprocessing, vocabulary và giá trị knots trên training portion. CV S3/S4 sau bước chọn quantile không phải ước lượng độc lập sau tuning; không dùng test trong bước này.

**Vì sao thử interaction?** Cùng số giờ AI có thể đi cùng kết quả khác nhau tùy mục đích sử dụng. Nhân các dummy Use Case với H và từng hinge cho phép đường theo H khác nhau giữa các nhóm, nhưng cũng thêm 16 hệ số. Run đã lưu: CV RMSE S3 0,157425 → S4 0,149997; validation 0,157403 → 0,149762. Đây là cải thiện dự đoán, không chứng minh Use Case gây ra khác biệt.


In [ ]:
class SplineUseCaseFeatures(BaseEstimator, TransformerMixin):
    """S3 design plus products with fold-fitted, reference-coded Use Case dummies."""
    def __init__(self, pre_gpa_quantile=0.75):
        self.pre_gpa_quantile = pre_gpa_quantile

    def fit(self, X, y=None):
        self.base_ = Pipeline([('preprocessing', clone(preprocessing)),
            ('hinge', QuantileHinges(quantiles=(0.25, 0.5, 0.75))),
            ('pre_gpa_hinge', QuantileHinges(quantiles=(self.pre_gpa_quantile,), feature_index=P_INDEX))])
        design = self.base_.fit_transform(X, y)
        prep_names = list(self.base_.named_steps['preprocessing'].get_feature_names_out())
        self.use_indices_ = [i for i, name in enumerate(prep_names)
                             if name.startswith('categorical__Primary_Use_Case_')]
        self.basis_indices_ = [prep_names.index('numeric__Weekly_GenAI_Hours'),
                              len(prep_names), len(prep_names)+1, len(prep_names)+2]
        base_names = [*prep_names, 'hinge__AI_Hours_q25', 'hinge__AI_Hours_q50',
                      'hinge__AI_Hours_q75', 'hinge__Pre_GPA']
        self.names_ = base_names + [base_names[b] + ' × ' + prep_names[u]
                                   for u in self.use_indices_ for b in self.basis_indices_]
        encoder = self.base_.named_steps['preprocessing'].named_transformers_['categorical'].named_steps['encode']
        self.use_levels_ = encoder.categories_[CATEGORICAL.index('Primary_Use_Case')].tolist()
        self.reference_ = self.use_levels_[0]
        if design.shape[1] != len(base_names):
            raise ValueError('S3 feature names không khớp design.')
        return self

    def transform(self, X):
        check_is_fitted(self, 'base_')
        design = self.base_.transform(X)
        # ponytail: chỉ spline AI Hours × Use Case; mở rộng khi có thí nghiệm riêng.
        products = [design[:, b] * design[:, u]
                    for u in self.use_indices_ for b in self.basis_indices_]
        result = np.column_stack([design, *products])
        if not np.isfinite(result).all():
            raise ValueError('S4 design chứa giá trị không hữu hạn.')
        return result

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, 'names_')
        return np.asarray(self.names_, dtype=object)

s4 = Pipeline([('features', SplineUseCaseFeatures(pre_gpa_quantile=s3_quantile)),
               ('ols', LinearRegression())])

# Check trên fixture: giữ nguyên S3 và từng tích có đúng giá trị tính tay.
f4_check = SplineUseCaseFeatures(pre_gpa_quantile=s3_quantile).fit(fixture)
baseline_design = f4_check.base_.transform(fixture)
s4_design = f4_check.transform(fixture)
np.testing.assert_allclose(s4_design[:, :baseline_design.shape[1]], baseline_design)
for j, (u, b) in enumerate((u, b) for u in f4_check.use_indices_ for b in f4_check.basis_indices_):
    np.testing.assert_allclose(s4_design[:, baseline_design.shape[1]+j],
                               baseline_design[:, u] * baseline_design[:, b])
assert s4_design.shape[1] == baseline_design.shape[1] + 4 * len(f4_check.use_indices_)
reference_rows = np.flatnonzero(fixture['Primary_Use_Case'].eq(f4_check.reference_))
assert np.all(s4_design[reference_rows, baseline_design.shape[1]:] == 0)
unknown = fixture.iloc[[0]].copy()
unknown['Primary_Use_Case'] = 'UNSEEN_TEST_CATEGORY'
original_knots = f4_check.base_.named_steps['hinge'].knots_.copy()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    unknown_design = f4_check.transform(unknown)
assert any('unknown' in str(w.message).lower() for w in caught)
assert np.all(unknown_design[:, baseline_design.shape[1]:] == 0)
np.testing.assert_array_equal(f4_check.base_.named_steps['hinge'].knots_, original_knots)
print('PASS: S4 hierarchy, product values, term count, reference, unknown category and fixed knots.')

In [ ]:
s4_records = []
for fold, (fit_idx, score_idx) in enumerate(folds, start=1):
    fitted = clone(s4).fit(X.iloc[fit_idx], y.iloc[fit_idx])
    feature_step = fitted.named_steps['features']
    base = feature_step.base_
    fit_design = base.named_steps['preprocessing'].transform(X.iloc[fit_idx])
    knots = base.named_steps['hinge'].knots_
    p_knot = float(base.named_steps['pre_gpa_hinge'].knots_[0])
    np.testing.assert_allclose(knots, np.quantile(fit_design[:, 0], [0.25, 0.5, 0.75]))
    assert np.isclose(p_knot, np.quantile(fit_design[:, P_INDEX], s3_quantile))
    prediction = fitted.predict(X.iloc[score_idx])
    s4_records.append({'model': 'S4', 'fold': fold, 'n': len(score_idx),
        'pre_gpa_knot': p_knot, 'terms': len(feature_step.get_feature_names_out()),
        **{f'hours_knot_{j}': float(k) for j,k in enumerate(knots,1)},
        **metrics(y.iloc[score_idx], prediction)})
s4_cv_results = pd.DataFrame(s4_records)
s4_cv_summary = s4_cv_results[['mae', 'mse', 'rmse', 'r2']].agg(['mean', 'std']).T
s4_cv_summary.columns = ['cv_mean', 'cv_sd']
s4.fit(X, y)
s4_val_prediction = s4.predict(val[INPUTS])
s4_validation_metrics = metrics(val[TARGET], s4_val_prediction)
f4 = s4.named_steps['features']
s4_feature_names = list(f4.get_feature_names_out())
assert len(s4_feature_names) == len(s3_feature_names) + 4 * len(f4.use_indices_)
assert len(f4.use_levels_) != 5 or len(s4_feature_names) == 39
assert np.isfinite(s4_val_prediction).all()
s4_coefficients = pd.DataFrame({'feature': ['intercept', *s4_feature_names],
    'coefficient': [s4.named_steps['ols'].intercept_, *s4.named_steps['ols'].coef_]})
s4_hinge_details = {'pre_gpa_quantile': s3_quantile,
    'pre_gpa_knot': float(f4.base_.named_steps['pre_gpa_hinge'].knots_[0]),
    'hours_knots': f4.base_.named_steps['hinge'].knots_.tolist(),
    'use_case_reference': f4.reference_, 'use_case_levels': f4.use_levels_}
comparison = pd.concat([comparison, pd.DataFrame([{'model': 'S4', 'terms': len(s4_feature_names),
    **{f'cv_{metric}_mean': float(s4_cv_summary.loc[metric, 'cv_mean']) for metric in s4_cv_summary.index},
    **{f'cv_{metric}_sd': float(s4_cv_summary.loc[metric, 'cv_sd']) for metric in s4_cv_summary.index},
    **{f'val_{metric}': value for metric, value in s4_validation_metrics.items()}}])], ignore_index=True)
display(s4_cv_results)
display(comparison)
print('S4 Use Case reference:', f4.reference_)
print('Test không được chạy trong bước development này.')

## 13. S5: giữ S4 + hinges cho ba numeric scores

Các knots cố định theo phân nhóm đã yêu cầu, không tuning hoặc tính từ validation/test:

| Biến | Nhóm giá trị nguyên | Knots | Số hinge thêm |
|---|---|---|---|
| Tool_Diversity | 1 / 2–5 | 1,5 | 1 |
| Perceived_AI_Dependency | 1–4 / 5–8 / 9–10 | 4,5 và 8,5 | 2 |
| Anxiety_Level_During_Exams | 1–4 / 5–8 / 9–10 | 4,5 và 8,5 | 2 |

Dùng mốc giữa hai số nguyên để tách đúng nhóm. Ví dụ hinge Tool = max(0, Tool−1,5). Đặt knot Tool=1 ở giá trị nhỏ nhất sẽ tạo cột Tool−1 phụ thuộc tuyến tính vào Tool và intercept.

S5 thêm 5 terms, dự kiến **44 predictor terms**. Giữ nguyên các biến gốc và tất cả interactions của S4; không thêm interactions cho các score hinges mới. Các đoạn vẫn liên tục, không thay score bằng category/group mean. Tool=1 chỉ có một mức quan sát ở đoạn đầu nên độ dốc đoạn đó phụ thuộc giả định nối tuyến tính, không có biến thiên trong nhóm để diễn giải thực nghiệm.

Cấu hình này được đề xuất sau khi xem development plots; kết quả vẫn mang tính exploratory. Giữ quantile Pre GPA từ S3, fit lại mọi thông số học được trong mỗi training fold. Không dùng test trong bước này.

**Vì sao thử các hinge này?** Sau khi xem đồ thị train, các nhóm được người thực hiện chọn để cho phép độ dốc khác nhau giữa các mức score. Đây là giả thuyết thăm dò, không phải ngưỡng đã được xác nhận. Run đã lưu: CV RMSE S4 0,149997 → S5 0,149919; validation 0,149762 → 0,149592. Lợi ích nhỏ so với 5 hệ số thêm; giữ S5 để tiếp tục thử S6 theo yêu cầu, chưa khẳng định từng hinge đều có ích.


In [ ]:
SCORE_KNOTS = {'Tool_Diversity': (1.5,),
               'Perceived_AI_Dependency': (4.5, 8.5),
               'Anxiety_Level_During_Exams': (4.5, 8.5)}

class FixedScoreHinges(BaseEstimator, TransformerMixin):
    """Append the five explicitly chosen hinges to the complete S4 matrix."""
    def fit(self, X, y=None):
        X = check_array(X, dtype=float)
        self.n_features_in_ = X.shape[1]
        self.indices_ = [NUMERIC.index(column) for column, knots in SCORE_KNOTS.items() for _ in knots]
        self.knots_ = np.array([k for knots in SCORE_KNOTS.values() for k in knots])
        if max(self.indices_) >= X.shape[1]:
            raise ValueError('S5 input thiếu các numeric score columns.')
        self.names_ = [f'hinge__{column}_above_{k:g}' for column, knots in SCORE_KNOTS.items() for k in knots]
        return self

    def transform(self, X):
        check_is_fitted(self, 'knots_')
        X = check_array(X, dtype=float)
        if X.shape[1] != self.n_features_in_:
            raise ValueError('Số cột S5 khác dữ liệu fit.')
        return np.column_stack([X, np.maximum(0, X[:, self.indices_] - self.knots_)])

s5 = Pipeline([('features', SplineUseCaseFeatures(pre_gpa_quantile=s3_quantile)),
               ('score_hinges', FixedScoreHinges()), ('ols', LinearRegression())])
# Check tính tay tại các mức sát ranh giới, giữ nguyên toàn bộ S4.
score_toy = np.zeros((3, len(s4_feature_names)))
score_toy[:, NUMERIC.index('Tool_Diversity')] = [1, 2, 5]
score_toy[:, NUMERIC.index('Perceived_AI_Dependency')] = [4, 5, 9]
score_toy[:, NUMERIC.index('Anxiety_Level_During_Exams')] = [8, 9, 10]
score_check = FixedScoreHinges().fit(score_toy)
augmented = score_check.transform(score_toy)
np.testing.assert_array_equal(augmented[:, :-5], score_toy)
np.testing.assert_allclose(augmented[:, -5:], [[0., 0., 0., 3.5, 0.],
    [0.5, 0.5, 0., 4.5, 0.5], [3.5, 4.5, 0.5, 5.5, 1.5]])
assert augmented.shape[1] == len(s4_feature_names) + 5
score_check.transform(score_toy * 2)
np.testing.assert_array_equal(score_check.knots_, [1.5, 4.5, 8.5, 4.5, 8.5])
print('PASS: S5 five hinge values, unchanged S4 features, fixed knots.')

In [ ]:
s5_records = []
for fold, (fit_idx, score_idx) in enumerate(folds, start=1):
    fitted = clone(s5).fit(X.iloc[fit_idx], y.iloc[fit_idx])
    base = fitted.named_steps['features'].base_
    fit_design = base.named_steps['preprocessing'].transform(X.iloc[fit_idx])
    hours_knots = base.named_steps['hinge'].knots_
    p_knot = float(base.named_steps['pre_gpa_hinge'].knots_[0])
    np.testing.assert_allclose(hours_knots, np.quantile(fit_design[:, 0], [0.25, 0.5, 0.75]))
    assert np.isclose(p_knot, np.quantile(fit_design[:, P_INDEX], s3_quantile))
    np.testing.assert_array_equal(fitted.named_steps['score_hinges'].knots_, [1.5, 4.5, 8.5, 4.5, 8.5])
    prediction = fitted.predict(X.iloc[score_idx])
    s5_records.append({'model': 'S5', 'fold': fold, 'n': len(score_idx), 'pre_gpa_knot': p_knot,
        **{f'hours_knot_{j}': float(k) for j,k in enumerate(hours_knots,1)},
        **metrics(y.iloc[score_idx], prediction)})
s5_cv_results = pd.DataFrame(s5_records)
s5_cv_summary = s5_cv_results[['mae', 'mse', 'rmse', 'r2']].agg(['mean', 'std']).T
s5_cv_summary.columns = ['cv_mean', 'cv_sd']
s5.fit(X, y)
s5_val_prediction = s5.predict(val[INPUTS])
s5_validation_metrics = metrics(val[TARGET], s5_val_prediction)
s5_feature_names = [*s5.named_steps['features'].get_feature_names_out(),
                    *s5.named_steps['score_hinges'].names_]
assert len(s5_feature_names) == len(s4_feature_names) + 5
assert np.isfinite(s5_val_prediction).all()
s5_coefficients = pd.DataFrame({'feature': ['intercept', *s5_feature_names],
    'coefficient': [s5.named_steps['ols'].intercept_, *s5.named_steps['ols'].coef_]})
s5_hinge_details = {**s4_hinge_details, 'score_knots': SCORE_KNOTS}
comparison = pd.concat([comparison, pd.DataFrame([{'model': 'S5', 'terms': len(s5_feature_names),
    **{f'cv_{metric}_mean': float(s5_cv_summary.loc[metric, 'cv_mean']) for metric in s5_cv_summary.index},
    **{f'cv_{metric}_sd': float(s5_cv_summary.loc[metric, 'cv_sd']) for metric in s5_cv_summary.index},
    **{f'val_{metric}': value for metric, value in s5_validation_metrics.items()}}])], ignore_index=True)
display(s5_cv_results)
display(comparison)
display(s5_coefficients.tail(5))
print('Test không được chạy trong bước development này.')

## 14. S6: Use Case × Prompt Skill

**S6 = S5 + tất cả Uᵢ × Kⱼ**, với Uᵢ là 4 dummy Use Case, Kⱼ là 2 dummy Prompt Skill. Thêm 8 terms, tổng 52 predictor terms khi đủ levels. Giữ đủ main effects để bảo toàn hierarchy.

Từng term bằng 1 khi sinh viên đồng thời thuộc hai mức tương ứng, ngược lại bằng 0. Nó cho phép chênh lệch GPA dự đoán giữa các mức Prompt Skill phụ thuộc vào Use Case, **không tự tạo thêm tương tác AI Hours × Prompt Skill**.

Reference, vocabulary và mọi thông số học được đều fit trong từng training fold. Giữ quantile Pre GPA từ S3 và knots score cố định từ S5. Đây là mở rộng exploratory sau các thí nghiệm train CV trước đó. Bước development này chỉ dùng train/CV và validation; kết quả test S6 nằm ở phần cuối.

**Vì sao thử interaction này?** Chênh lệch dự đoán giữa các mức Prompt Skill có thể khác theo Use Case. Dummy × dummy biểu diễn từng tổ hợp so với các mức tham chiếu. Run đã lưu: CV RMSE S5 0,149919 → S6 0,146556; validation 0,149592 → 0,146712. Người thực hiện chọn S6 làm model cuối sau so sánh; kết quả không chứng minh quan hệ nhân quả.


In [ ]:
class UseCaseSkillFeatures(BaseEstimator, TransformerMixin):
    """Preserve S5 and add reference-coded Use Case × Prompt Skill products."""
    def __init__(self, pre_gpa_quantile=0.75):
        self.pre_gpa_quantile = pre_gpa_quantile

    def fit(self, X, y=None):
        self.base_ = Pipeline([('features', SplineUseCaseFeatures(self.pre_gpa_quantile)),
                               ('score_hinges', FixedScoreHinges())])
        design = self.base_.fit_transform(X, y)
        base_names = [*self.base_.named_steps['features'].get_feature_names_out(),
                      *self.base_.named_steps['score_hinges'].names_]
        uses = [i for i,name in enumerate(base_names) if name.startswith('categorical__Primary_Use_Case_')]
        skills = [i for i,name in enumerate(base_names) if name.startswith('categorical__Prompt_Engineering_Skill_')]
        self.pairs_ = [(u,k) for u in uses for k in skills]
        self.names_ = base_names + [base_names[u] + ' × ' + base_names[k] for u,k in self.pairs_]
        if design.shape[1] != len(base_names):
            raise ValueError('S6 base feature names không khớp design.')
        self.base_width_ = len(base_names)
        return self

    def transform(self, X):
        check_is_fitted(self, 'pairs_')
        design = self.base_.transform(X)
        products = [design[:,u] * design[:,k] for u,k in self.pairs_]
        return np.column_stack([design, *products])

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, 'names_')
        return np.asarray(self.names_, dtype=object)

s6 = Pipeline([('features', UseCaseSkillFeatures(pre_gpa_quantile=s3_quantile)),
               ('ols', LinearRegression())])
# Runnable check: fixture A/B → một term interaction; giữ nguyên toàn bộ S5.
f6_check = UseCaseSkillFeatures(s3_quantile).fit(fixture)
baseline6 = f6_check.base_.transform(fixture)
augmented6 = f6_check.transform(fixture)
np.testing.assert_allclose(augmented6[:, :f6_check.base_width_], baseline6)
assert len(f6_check.pairs_) == 1
for j,(u,k) in enumerate(f6_check.pairs_):
    np.testing.assert_array_equal(augmented6[:, f6_check.base_width_+j], baseline6[:,u]*baseline6[:,k])
np.testing.assert_array_equal(augmented6[:, -1], [0., 1., 0., 1., 0., 1.])
unknown6 = fixture.iloc[[1]].copy()
unknown6['Primary_Use_Case'] = 'UNSEEN_TEST_CATEGORY'
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    unseen6 = f6_check.transform(unknown6)
assert any('unknown' in str(w.message).lower() for w in caught)
assert np.all(unseen6[:, f6_check.base_width_:] == 0)
print('PASS: S6 preserves S5, correct dummy products, references and unseen fallback.')

In [ ]:
s6_records = []
for fold, (fit_idx, score_idx) in enumerate(folds, start=1):
    fitted = clone(s6).fit(X.iloc[fit_idx], y.iloc[fit_idx])
    base = fitted.named_steps['features'].base_.named_steps['features'].base_
    fit_design = base.named_steps['preprocessing'].transform(X.iloc[fit_idx])
    hours_knots = base.named_steps['hinge'].knots_
    p_knot = float(base.named_steps['pre_gpa_hinge'].knots_[0])
    np.testing.assert_allclose(hours_knots, np.quantile(fit_design[:,0], [0.25,0.5,0.75]))
    assert np.isclose(p_knot, np.quantile(fit_design[:,P_INDEX], s3_quantile))
    prediction = fitted.predict(X.iloc[score_idx])
    s6_records.append({'model': 'S6', 'fold': fold, 'n': len(score_idx),
        'pre_gpa_knot': p_knot,
        **{f'hours_knot_{j}': float(k) for j,k in enumerate(hours_knots,1)},
        **metrics(y.iloc[score_idx], prediction)})
s6_cv_results = pd.DataFrame(s6_records)
s6_cv_summary = s6_cv_results[['mae','mse','rmse','r2']].agg(['mean','std']).T
s6_cv_summary.columns = ['cv_mean','cv_sd']
s6.fit(X, y)
s6_val_prediction = s6.predict(val[INPUTS])
s6_validation_metrics = metrics(val[TARGET], s6_val_prediction)
f6 = s6.named_steps['features']
s6_feature_names = list(f6.get_feature_names_out())
assert len(s6_feature_names) == len(s5_feature_names) + len(f6.pairs_)
assert len(f6.pairs_) == 8  # đủ 5 Use Cases × 3 Prompt Skills trong train hiện tại
assert np.isfinite(s6_val_prediction).all()
s6_coefficients = pd.DataFrame({'feature': ['intercept', *s6_feature_names],
    'coefficient': [s6.named_steps['ols'].intercept_, *s6.named_steps['ols'].coef_]})
s6_hinge_details = {**s5_hinge_details, 'extra_interactions': [s6_feature_names[-8:]]}
comparison = pd.concat([comparison, pd.DataFrame([{'model': 'S6', 'terms': len(s6_feature_names),
    **{f'cv_{metric}_mean': float(s6_cv_summary.loc[metric,'cv_mean']) for metric in s6_cv_summary.index},
    **{f'cv_{metric}_sd': float(s6_cv_summary.loc[metric,'cv_sd']) for metric in s6_cv_summary.index},
    **{f'val_{metric}': value for metric,value in s6_validation_metrics.items()}}])],ignore_index=True)
display(s6_cv_results)
display(comparison)
display(s6_coefficients.tail(8))
print('Không chạy test trong bước development S6.')

## 15. Đường dự đoán và sai số theo khoảng AI Hours

Đường dự đoán S0–S6 dùng cùng một profile thực từ hàng đầu của train, chỉ thay AI Hours trên miền min–max train. Các đường thể hiện dự đoán có điều kiện tại profile này, không phải GPA trung bình toàn bộ sinh viên hoặc tác động nhân quả. Nét chấm đánh dấu ba knots S2; knot S1 trùng knot giữa.

Sai số được tính trên **toàn bộ validation** với inputs thực tế của từng sinh viên. Các khoảng dùng knots S2 đã học từ train, không tính quantile mới trên validation. Luôn báo n; nhóm n<30 cần thận trọng. Bảng RMSE CV theo fold được ghép cặp vì các model dùng cùng folds.

**Vì sao xem thêm sai số theo nhóm?** RMSE tổng thể có thể che khuất nhóm dự đoán kém. Đây là phân tích validation để hiểu model, không dùng các bảng này làm bằng chứng test độc lập.


In [ ]:
profile = train[INPUTS].iloc[0].copy()
display(profile.rename('fixed_profile').to_frame())
hours_grid = np.unique(np.r_[np.linspace(train['Weekly_GenAI_Hours'].min(),
    train['Weekly_GenAI_Hours'].max(), 300), s1_knot, s2_knots])
profiles = pd.DataFrame([profile.to_dict()] * len(hours_grid))
profiles['Weekly_GenAI_Hours'] = hours_grid
curve_data = pd.DataFrame({'Weekly_GenAI_Hours': hours_grid})
fig_curves, ax = plt.subplots(figsize=(9, 4.5))
for name, model in [('S0', s0), ('S1', s1), ('S2', s2), ('S3', s3), ('S4', s4), ('S5', s5), ('S6', s6)]:
    curve_data[name] = model.predict(profiles)
    ax.plot(hours_grid, curve_data[name], label=name, linewidth=2)
for j, knot in enumerate(s2_knots, start=1):
    ax.axvline(knot, color='gray', linestyle=':', alpha=0.7)
    ax.text(knot, 1.01, f'k{j}={knot:.2f}', transform=ax.get_xaxis_transform(),
            ha='center', fontsize=9)
ax.set(xlabel='Weekly GenAI Hours', ylabel='Predicted Post-Semester GPA',
       title='S0–S6: same train profile, varying AI Hours')
ax.legend()
ax.grid(alpha=0.2)
fig_curves.tight_layout()
display(fig_curves)
plt.close(fig_curves)

bounds = [-np.inf, *s2_knots, np.inf]
segment_labels = [f'H ≤ {s2_knots[0]:.2f}',
    f'{s2_knots[0]:.2f} < H ≤ {s2_knots[1]:.2f}',
    f'{s2_knots[1]:.2f} < H ≤ {s2_knots[2]:.2f}', f'H > {s2_knots[2]:.2f}']
segments = pd.cut(val['Weekly_GenAI_Hours'], bins=bounds, labels=segment_labels,
                  right=True).cat.add_categories(['Missing H (imputed)']).fillna('Missing H (imputed)')
subgroup_rows = []
for name, prediction in [('S0', val_prediction), ('S1', s1_val_prediction), ('S2', s2_val_prediction), ('S3', s3_val_prediction), ('S4', s4_val_prediction), ('S5', s5_val_prediction), ('S6', s6_val_prediction)]:
    for label in segments.cat.categories:
        mask = (segments == label).to_numpy()
        count = int(mask.sum())
        if count:
            errors = val[TARGET].to_numpy()[mask] - np.asarray(prediction)[mask]
            subgroup_rows.append({'model': name, 'segment': label, 'n': count,
                'mae': float(np.mean(np.abs(errors))), 'rmse': float(np.sqrt(np.mean(errors**2))),
                'warning': 'n<30' if count < 30 else ''})
subgroup_errors = pd.DataFrame(subgroup_rows)
assert subgroup_errors.groupby('model')['n'].sum().eq(len(val)).all()
# Independent check: subgroup squared errors must reconstruct overall validation MSE.
for name, scores in [('S0', validation_metrics), ('S1', s1_validation_metrics), ('S2', s2_validation_metrics), ('S3', s3_validation_metrics), ('S4', s4_validation_metrics), ('S5', s5_validation_metrics), ('S6', s6_validation_metrics)]:
    group = subgroup_errors[subgroup_errors['model'] == name]
    assert np.isclose(np.average(group['rmse']**2, weights=group['n']), scores['mse'])
display(subgroup_errors)

fig_errors, ax = plt.subplots(figsize=(9, 4.5))
subgroup_errors.pivot(index='segment', columns='model', values='rmse').reindex(
    [label for label in segments.cat.categories if (segments == label).any()]).plot.bar(ax=ax, rot=15)
ax.set(xlabel='AI Hours segments from train knots', ylabel='Validation RMSE',
       title='S0–S6: validation errors by AI Hours segment')
ax.grid(axis='y', alpha=0.2)
fig_errors.tight_layout()
display(fig_errors)
plt.close(fig_errors)

paired_rmse = cv_results[['fold', 'rmse']].rename(columns={'rmse': 'S0'}).merge(
    s1_cv_results[['fold', 'rmse']].rename(columns={'rmse': 'S1'}), on='fold').merge(
    s2_cv_results[['fold', 'rmse']].rename(columns={'rmse': 'S2'}), on='fold')
paired_rmse['S1_minus_S0'] = paired_rmse['S1'] - paired_rmse['S0']
paired_rmse['S2_minus_S1'] = paired_rmse['S2'] - paired_rmse['S1']
display(paired_rmse)
slope_table = pd.DataFrame({'segment': segment_labels, 'slope_GPA_per_hour': s2_segment_slopes})
display(slope_table)
print('Độ dốc dương: GPA dự đoán tăng theo H; âm: giảm theo H, giữ các input khác cố định.')
print('Các slope mô tả model; chưa có kiểm định ý nghĩa thống kê hoặc bằng chứng nhân quả.')

paired_rmse = paired_rmse.merge(s3_cv_results[['fold', 'rmse']].rename(columns={'rmse': 'S3'}), on='fold')
paired_rmse['S3_minus_S2'] = paired_rmse['S3'] - paired_rmse['S2']
display(paired_rmse)
p_grid = np.unique(np.r_[np.linspace(train['Pre_Semester_GPA'].min(),
    train['Pre_Semester_GPA'].max(), 250), s3_p_knot])
p_profiles = pd.DataFrame([profile.to_dict()] * len(p_grid))
p_profiles['Pre_Semester_GPA'] = p_grid
pre_gpa_curves = pd.DataFrame({'Pre_Semester_GPA': p_grid})
fig_pre_gpa, ax = plt.subplots(figsize=(9, 4.5))
for name, model in [('S2', s2), ('S3', s3), ('S4', s4), ('S5', s5), ('S6', s6)]:
    pre_gpa_curves[name] = model.predict(p_profiles)
    ax.plot(p_grid, pre_gpa_curves[name], label=name, linewidth=2)
ax.axvline(s3_p_knot, color='gray', linestyle=':', label=f'Pre GPA knot={s3_p_knot:.3f}')
ax.set(xlabel='Pre-Semester GPA', ylabel='Predicted Post-Semester GPA',
       title='S2–S6: same train profile, varying Pre GPA')
ax.legend()
ax.grid(alpha=0.2)
fig_pre_gpa.tight_layout()
display(fig_pre_gpa)
plt.close(fig_pre_gpa)

paired_rmse = paired_rmse.merge(s4_cv_results[['fold', 'rmse']].rename(columns={'rmse': 'S4'}), on='fold')
paired_rmse['S4_minus_S3'] = paired_rmse['S4'] - paired_rmse['S3']
display(paired_rmse)

# Cùng profile và lưới H; chỉ đổi Use Case giữa các đường.
fig_use_cases, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharex=True, sharey=True)
use_curve_rows, slope_rows = [], []
for ax, (name, model) in zip(axes, [('S3', s3), ('S4', s4)]):
    for use_case in f4.use_levels_:
        group_profiles = profiles.copy()
        group_profiles['Primary_Use_Case'] = use_case
        pred = model.predict(group_profiles)
        ax.plot(hours_grid, pred, label=use_case, linewidth=1.8)
        use_curve_rows.extend({'model': name, 'use_case': use_case, 'hours': float(h),
                               'prediction': float(v)} for h,v in zip(hours_grid, pred))
    for knot in s2_knots:
        ax.axvline(knot, color='gray', linestyle=':', alpha=0.5)
    ax.set(title=f'{name}: varying AI Hours by Use Case', xlabel='Weekly GenAI Hours')
    ax.grid(alpha=0.2)
axes[0].set_ylabel('Predicted Post-Semester GPA')
axes[1].legend(fontsize=8)
fig_use_cases.tight_layout()
display(fig_use_cases)
plt.close(fig_use_cases)
use_case_curves = pd.DataFrame(use_curve_rows)

# Slope tính từ hệ số, đối chiếu finite difference bên trong từng đoạn.
s4_coef = s4.named_steps['ols'].coef_
base_width = len(s3_feature_names)
knots4 = np.asarray(s4_hinge_details['hours_knots'])
probe_hours = [(train['Weekly_GenAI_Hours'].min()+knots4[0])/2,
               (knots4[0]+knots4[1])/2, (knots4[1]+knots4[2])/2,
               (knots4[2]+train['Weekly_GenAI_Hours'].max())/2]
for use_case in f4.use_levels_:
    group_profile = profile.to_dict() | {'Primary_Use_Case': use_case}
    row_design = f4.base_.transform(pd.DataFrame([group_profile]))[0]
    basis_coef = s4_coef[f4.basis_indices_].copy()
    for j, u in enumerate(f4.use_indices_):
        basis_coef += row_design[u] * s4_coef[base_width+4*j:base_width+4*j+4]
    slopes = basis_coef[0] + np.r_[0., np.cumsum(basis_coef[1:])]
    for segment, h, slope in zip(segment_labels, probe_hours, slopes):
        pair = pd.DataFrame([group_profile | {'Weekly_GenAI_Hours': h},
                             group_profile | {'Weekly_GenAI_Hours': h+1e-4}])
        assert np.isclose(np.diff(s4.predict(pair))[0]/1e-4, slope, atol=1e-8)
        slope_rows.append({'use_case': use_case, 'segment': segment, 'slope_GPA_per_hour': float(slope)})
    for knot in knots4:
        pair = pd.DataFrame([group_profile | {'Weekly_GenAI_Hours': knot-1e-7},
                             group_profile | {'Weekly_GenAI_Hours': knot+1e-7}])
        assert abs(np.diff(s4.predict(pair))[0]) < 1e-5
s4_group_slopes = pd.DataFrame(slope_rows)
display(s4_group_slopes)
use_error_rows = []
for name, pred in [('S3', s3_val_prediction), ('S4', s4_val_prediction), ('S5', s5_val_prediction), ('S6', s6_val_prediction)]:
    for use_case in sorted(val['Primary_Use_Case'].dropna().unique()):
        mask = val['Primary_Use_Case'].eq(use_case).to_numpy()
        errors = val[TARGET].to_numpy()[mask] - np.asarray(pred)[mask]
        use_error_rows.append({'model': name, 'use_case': use_case, 'n': int(mask.sum()),
            'mae': float(np.abs(errors).mean()), 'rmse': float(np.sqrt(np.mean(errors**2))),
            'warning': 'n<30' if mask.sum() < 30 else ''})
use_case_errors = pd.DataFrame(use_error_rows)
assert use_case_errors.groupby('model')['n'].sum().eq(val['Primary_Use_Case'].notna().sum()).all()
display(use_case_errors)
print('PASS: S4 conditional slopes match predictions; continuous at knots.')

paired_rmse = paired_rmse.merge(s5_cv_results[['fold', 'rmse']].rename(columns={'rmse': 'S5'}), on='fold')
paired_rmse['S5_minus_S4'] = paired_rmse['S5'] - paired_rmse['S4']
display(paired_rmse)

score_figures, score_curve_rows, score_slope_rows, score_error_rows = {}, [], [], []
for column, knots in SCORE_KNOTS.items():
    low, high = train[column].min(), train[column].max()
    grid = np.unique(np.r_[np.linspace(low, high, 200), knots, np.arange(low, high+1)])
    group_profiles = pd.DataFrame([profile.to_dict()] * len(grid))
    group_profiles[column] = grid
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for name, model in [('S4', s4), ('S5', s5), ('S6', s6)]:
        pred = model.predict(group_profiles)
        ax.plot(grid, pred, label=name, linewidth=2)
        score_curve_rows.extend({'feature': column, 'model': name, 'value': float(v),
            'prediction': float(yp)} for v,yp in zip(grid, pred))
    for knot in knots:
        ax.axvline(knot, color='gray', linestyle=':', alpha=0.7)
    ax.set(xlabel=column, ylabel='Predicted Post_Semester_GPA',
           title=f'S4–S5: varying {column}, same train profile')
    ax.set_xticks(np.arange(low, high+1))
    ax.legend()
    ax.grid(alpha=0.2)
    fig.tight_layout()
    display(fig)
    plt.close(fig)
    score_figures[column] = fig
    # Slopes from coefficients, validated against predictions inside each interval.
    base_coef = s5.named_steps['ols'].coef_[NUMERIC.index(column)]
    hinge_names = [f'hinge__{column}_above_{k:g}' for k in knots]
    changes = [s5.named_steps['ols'].coef_[s5_feature_names.index(name)] for name in hinge_names]
    slopes = base_coef + np.r_[0., np.cumsum(changes)]
    edges = [low, *knots, high]
    for j, slope in enumerate(slopes):
        probe = (edges[j] + edges[j+1]) / 2
        pair = pd.DataFrame([profile.to_dict() | {column: probe},
                             profile.to_dict() | {column: probe+1e-4}])
        assert np.isclose(np.diff(s5.predict(pair))[0]/1e-4, slope, atol=1e-8)
        score_slope_rows.append({'feature': column, 'segment': j+1, 'slope': float(slope)})
    for knot in knots:
        pair = pd.DataFrame([profile.to_dict() | {column: knot-1e-7},
                             profile.to_dict() | {column: knot+1e-7}])
        assert abs(np.diff(s5.predict(pair))[0]) < 1e-5
    labels = ['1', '2–5'] if column == 'Tool_Diversity' else ['1–4', '5–8', '9–10']
    groups = pd.cut(val[column], [-np.inf, *knots, np.inf], labels=labels).cat.add_categories(['missing']).fillna('missing')
    for name, prediction in [('S4', s4_val_prediction), ('S5', s5_val_prediction), ('S6', s6_val_prediction)]:
        for label in groups.cat.categories:
            mask = groups.eq(label).to_numpy()
            if mask.any():
                errors = val[TARGET].to_numpy()[mask] - np.asarray(prediction)[mask]
                score_error_rows.append({'feature': column, 'model': name, 'group': label,
                    'n': int(mask.sum()), 'mae': float(np.abs(errors).mean()),
                    'rmse': float(np.sqrt(np.mean(errors**2))), 'warning': 'n<30' if mask.sum()<30 else ''})
score_curves = pd.DataFrame(score_curve_rows)
score_slopes = pd.DataFrame(score_slope_rows)
score_errors = pd.DataFrame(score_error_rows)
assert score_errors.groupby(['feature','model'])['n'].sum().eq(len(val)).all()
for column in SCORE_KNOTS:
    for name, metric in [('S4',s4_validation_metrics),('S5',s5_validation_metrics)]:
        g = score_errors[(score_errors['feature']==column)&(score_errors['model']==name)]
        assert np.isclose(np.average(g['rmse']**2,weights=g['n']),metric['mse'])
display(score_slopes)
display(score_errors)
print('Các đường giữa số nguyên là nội suy của model, không phải giá trị score quan sát.')
print('PASS: S5 score slopes, continuity and subgroup error reconstruction.')

paired_rmse = paired_rmse.merge(s6_cv_results[['fold','rmse']].rename(columns={'rmse':'S6'}),on='fold')
paired_rmse['S6_minus_S5'] = paired_rmse['S6'] - paired_rmse['S5']
display(paired_rmse)

skill_levels = sorted(train['Prompt_Engineering_Skill'].dropna().unique())
use_levels = f4.use_levels_
combo_profiles = pd.DataFrame([profile.to_dict() | {'Primary_Use_Case':u,'Prompt_Engineering_Skill':k}
                              for u in use_levels for k in skill_levels])
combo_predictions = combo_profiles[['Primary_Use_Case','Prompt_Engineering_Skill']].copy()
for name, model in [('S5',s5),('S6',s6)]:
    combo_predictions[name] = model.predict(combo_profiles)
fig_combo, axes = plt.subplots(1,2,figsize=(13,4.5),sharey=True)
vmin = combo_predictions[['S5','S6']].to_numpy().min()
vmax = combo_predictions[['S5','S6']].to_numpy().max()
for ax,name in zip(axes,['S5','S6']):
    matrix = combo_predictions.pivot(index='Primary_Use_Case',columns='Prompt_Engineering_Skill',values=name).reindex(index=use_levels,columns=skill_levels)
    heat = ax.imshow(matrix.to_numpy(),vmin=vmin,vmax=vmax,cmap='viridis',aspect='auto')
    for i in range(len(use_levels)):
        for j in range(len(skill_levels)):
            ax.text(j,i,f'{matrix.iloc[i,j]:.3f}',ha='center',va='center',color='white',fontsize=9)
    ax.set_xticks(range(len(skill_levels)),skill_levels)
    ax.set_yticks(range(len(use_levels)),use_levels)
    ax.set(title=f'{name}: predicted GPA, same profile',xlabel='Prompt Skill')
axes[0].set_ylabel('Use Case')
fig_combo.tight_layout()
display(fig_combo)
plt.close(fig_combo)
# Actual validation errors by both group labels; no target aggregate is used as an input.
combo_error_rows = []
labels = val[['Primary_Use_Case','Prompt_Engineering_Skill']].fillna('(missing)')
for name,prediction in [('S5',s5_val_prediction),('S6',s6_val_prediction)]:
    for (use,skill), indices in labels.groupby(['Primary_Use_Case','Prompt_Engineering_Skill']).indices.items():
        error = val[TARGET].to_numpy()[indices] - np.asarray(prediction)[indices]
        combo_error_rows.append({'model':name,'use_case':use,'prompt_skill':skill,'n':len(indices),
            'mae':float(np.abs(error).mean()),'rmse':float(np.sqrt(np.mean(error**2))),
            'warning':'n<30' if len(indices)<30 else ''})
combo_errors = pd.DataFrame(combo_error_rows)
assert combo_errors.groupby('model')['n'].sum().eq(len(val)).all()
for name,metric in [('S5',s5_validation_metrics),('S6',s6_validation_metrics)]:
    g=combo_errors[combo_errors['model']==name]
    assert np.isclose(np.average(g['rmse']**2,weights=g['n']),metric['mse'])
display(combo_errors)
print('Heatmaps là dự đoán cùng profile cố định, không phải GPA trung bình thực tế của các nhóm.')
print('PASS: S6 subgroup errors reconstruct overall validation MSE.')


## 16. Lưu kết quả của run

Mỗi run lưu model S6 fit trên train, metrics, các bảng so sánh và biểu đồ phục vụ báo cáo trong một thư mục riêng. S0–S5 có thể chạy lại từ notebook. Chỉ tải joblib do nhóm tạo và tin cậy.

Để tải lại S6, chạy các cell định nghĩa `QuantileHinges`, `SplineUseCaseFeatures`, `FixedScoreHinges`, `UseCaseSkillFeatures` trước. Year và Policy phải được map như cell đọc dữ liệu trước khi đưa vào model.


In [ ]:
run_id = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8]
OUT = ROOT / 'outputs/piecewise/runs' / run_id
OUT.mkdir(parents=True, exist_ok=False)

# Các bảng dùng để so sánh, giải thích mô hình và phân tích validation.
for name, table in {
    'comparison': comparison, 'paired_cv_rmse': paired_rmse,
    's3_knot_search': s3_knot_search, 's6_coefficients': s6_coefficients,
    'validation_errors_by_hours': subgroup_errors,
    's6_validation_errors_by_combo': combo_errors,
}.items():
    table.to_csv(OUT / f'{name}.csv', index=False)

figures = {
    'prediction_curves': fig_curves, 'validation_errors_by_hours': fig_errors,
    'pre_gpa_prediction_curves': fig_pre_gpa, 'use_case_prediction_curves': fig_use_cases,
    's6_use_case_prompt_skill': fig_combo,
    **{f'score_{column}': fig for column, fig in score_figures.items()},
}
for name, figure in figures.items():
    figure.savefig(OUT / f'{name}.png', dpi=180, bbox_inches='tight')
figure_dir = OUT / 'numeric_vs_gpa'
figure_dir.mkdir()
for column, figure in numeric_figures.items():
    figure.savefig(figure_dir / f'{column}.png', dpi=180, bbox_inches='tight')

model_path = OUT / 's6_train_only.joblib'
joblib.dump(s6, model_path)
s6_result = {'model': 'S6', 'seed': SEED, 'n_folds': N_FOLDS,
    'predictor_terms': len(s6_feature_names), 'fit_scope': 'train',
    'year_levels': YEAR_LEVELS, 'policy_levels': POLICY_LEVELS,
    'numeric_features': NUMERIC, 'categorical_features': CATEGORICAL,
    'counts': {name: len(frame) for name, frame in frames.items()},
    'split_sha256': hashes, 'cv': s6_cv_summary.to_dict(orient='index'),
    'validation': s6_validation_metrics, 'hinge': s6_hinge_details}
(OUT / 's6_metrics.json').write_text(json.dumps(s6_result, indent=2), encoding='utf-8')
np.testing.assert_allclose(joblib.load(model_path).predict(val[INPUTS].head(5)), s6_val_prediction[:5])
print('Đã lưu model S6, bảng và hình:', OUT)


## 17. Model cuối: S6

Chọn **S6**, gồm 52 predictor terms, fit trên train. Model và kết quả CV/validation được lưu theo run. File `outputs/piecewise/final_model.json` ghi đường dẫn artifact và SHA-256 để xác định đúng model đã chọn.

Cell dưới chỉ lưu lựa chọn S6 khi chưa có lựa chọn; nếu đã có thì giữ nguyên. Cell không refit hoặc đánh giá test.

**Lý do chọn:** S6 có CV và validation RMSE thấp hơn S5 trong run đã lưu, đồng thời là lựa chọn cuối của người thực hiện. S5 chỉ hơn S4 rất ít; có thể so sánh thêm S4 + interaction của S6 nếu cần đánh giá riêng giá trị của các score hinges, dùng development data.


In [ ]:
FINAL_MODEL_NAME = 'S6'
final_model_path = OUT / 's6_train_only.joblib'
final_metrics_path = OUT / 's6_metrics.json'
final_metrics = json.loads(final_metrics_path.read_text(encoding='utf-8'))
if final_metrics['model'] != FINAL_MODEL_NAME or final_metrics['fit_scope'] != 'train':
    raise ValueError('Artifact phải là S6 fit train-only.')
selection = {'selected_model': FINAL_MODEL_NAME, 'predictor_terms': final_metrics['predictor_terms'],
    'fit_scope': 'train', 'selected_by': 'user',
    'model_path': str(final_model_path.relative_to(ROOT)),
    'model_sha256': hashlib.sha256(final_model_path.read_bytes()).hexdigest(),
    'metrics_path': str(final_metrics_path.relative_to(ROOT)),
    'selected_at_utc': datetime.now(timezone.utc).isoformat()}
selection_path = ROOT / 'outputs/piecewise/final_model.json'
if selection_path.exists():
    print('Giữ nguyên model cuối đã chọn:', selection_path)
    print('Run mới được lưu riêng, không thay thế model đã đánh giá test.')
else:
    selection_path.write_text(json.dumps(selection, indent=2), encoding='utf-8')
    print('Model cuối:', FINAL_MODEL_NAME, '| terms:', selection['predictor_terms'])
    print('Artifact:', final_model_path)


## 18. Đánh giá S6 trên test

Dùng artifact S6 train-only đã chọn trong `final_model.json`; kiểm tra SHA-256 và lưu model lock trước khi đọc target test. Không refit trên validation/test và không clip dự đoán.

Mặc định `RUN_S6_TEST=False` để Run All không tự đánh giá lại. Bật có chủ đích sẽ đọc kết quả đã lưu nếu model/split khớp. 

Kết quả đã lưu: RMSE 0,145724, MAE 0,116090, R² 0,914158 trên 7.500 mẫu; model chỉ fit trên train.


In [ ]:
# Mặc định tắt; bật có chủ đích để đánh giá hoặc đọc kết quả đã lưu.
RUN_S6_TEST = False
TEST_OUT = ROOT / 'outputs/piecewise/s6_test'
if not RUN_S6_TEST:
    print('Không chạy test tự động. Kết quả S6 đã lưu tại:', TEST_OUT / 'metrics.json')
else:
    selected = json.loads((ROOT / 'outputs/piecewise/final_model.json').read_text(encoding='utf-8'))
    model_path = ROOT / selected['model_path']
    model_hash = hashlib.sha256(model_path.read_bytes()).hexdigest()
    if selected['selected_model'] != 'S6' or selected['fit_scope'] != 'train' or model_hash != selected['model_sha256']:
        raise ValueError('Model không khớp S6 train-only đã chọn.')
    source_metrics = json.loads((ROOT / selected['metrics_path']).read_text(encoding='utf-8'))
    meta = json.loads((DATA_ROOT / 'metadata.json').read_text(encoding='utf-8'))
    current_hashes = {split: hashlib.sha256((DATA_ROOT / split / 'dataset.csv').read_bytes()).hexdigest()
                      for split in ['train','validation','test']}
    if current_hashes != source_metrics['split_sha256']:
        raise ValueError('Split khác dữ liệu dùng cho S6.')
    if any(current_hashes[s] != meta['dataset_sha256'][f'{s}/dataset.csv'] for s in current_hashes):
        raise ValueError('Split hashes khác metadata.')
    TEST_OUT.mkdir(parents=True,exist_ok=True)
    lock_path = TEST_OUT / 'model_lock.json'
    metrics_path = TEST_OUT / 'metrics.json'
    if metrics_path.exists():
        results = json.loads(metrics_path.read_text(encoding='utf-8'))
        if results['model_sha256'] != model_hash or results['test_sha256'] != current_hashes['test']:
            raise RuntimeError('Đã có kết quả test cho model/split khác; không ghi đè.')
        print('Đọc kết quả đã lưu, không dự đoán lại test.')
    else:
        lock = {**selected,'test_sha256':current_hashes['test'],
                'locked_at_utc':datetime.now(timezone.utc).isoformat()}
        # Exclusive lock trước khi đọc test target; lỗi không âm thầm mở đường đánh giá lại.
        with lock_path.open('x',encoding='utf-8') as f:
            json.dump(lock,f,indent=2)
        model = joblib.load(model_path)
        test = pd.read_csv(DATA_ROOT / 'test/dataset.csv',dtype={'Student_ID':str})
        assignment_path = DATA_ROOT / 'assignments.csv'
        if hashlib.sha256(assignment_path.read_bytes()).hexdigest() != meta['assignment_sha256']:
            raise ValueError('Assignment hash khác metadata.')
        assignment = pd.read_csv(assignment_path,dtype={'Student_ID':str})
        missing = sorted(set(INPUTS+[TARGET,'Student_ID'])-set(test.columns))
        if missing:
            raise ValueError(f'Test thiếu cột {missing}')
        if test.Student_ID.isna().any() or not test.Student_ID.is_unique or set(test.Student_ID) != set(assignment.loc[assignment['split']=='test','Student_ID']):
            raise ValueError('Test IDs thiếu/trùng/khác assignment.')
        for column,mapping in [('Year_of_Study',YEAR_LEVELS),('Institutional_Policy',POLICY_LEVELS)]:
            unknown = set(test[column].dropna())-set(mapping)
            if unknown:
                raise ValueError(f'Test {column}: mức không xác định {unknown}')
            test[column] = test[column].map(mapping)
        for column in NUMERIC+[TARGET]:
            test[column] = pd.to_numeric(test[column],errors='raise')
            if np.isinf(test[column].to_numpy(dtype=float)).any():
                raise ValueError(f'Test {column}: infinity')
        if test[TARGET].isna().any() or not test[TARGET].between(0,4).all():
            raise ValueError('Test target thiếu hoặc ngoài [0,4].')
        if not test.Paid_Subscription.dropna().isin([0,1]).all():
            raise ValueError('Paid_Subscription phải là 0/1.')
        for column in CATEGORICAL:
            test[column] = test[column].map(lambda x:str(x) if pd.notna(x) else np.nan)
        prediction = model.predict(test[INPUTS])  # Không fit/refit hoặc clip.
        if not np.isfinite(prediction).all():
            raise ValueError('Test prediction không hữu hạn.')
        scores = metrics(test[TARGET],prediction)
        predictions = pd.DataFrame({'Student_ID':test.Student_ID,'actual':test[TARGET],
            'predicted':prediction,'residual':test[TARGET].to_numpy()-prediction})
        assert np.isclose(np.mean(predictions.residual**2),scores['mse'])
        assert np.isclose(np.mean(np.abs(predictions.residual)),scores['mae'])
        results = {'model':'S6','status':'executed','n':len(test),'fit_scope':'train',
            'predictor_terms':selected['predictor_terms'],'model_sha256':model_hash,
            'test_sha256':current_hashes['test'],**scores,'clipped':False,
            'pred_below_0':int((prediction<0).sum()),'pred_above_4':int((prediction>4).sum()),
            'completed_at_utc':datetime.now(timezone.utc).isoformat()}
        metrics_path.write_text(json.dumps(results,indent=2),encoding='utf-8')
    display(pd.Series(results,name='S6 test').to_frame())
    print('S6 test artifacts:',TEST_OUT)
